In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm import tqdm
import ast
import numpy as np

In [ ]:
# read market news data
df = pd.read_csv('market_news_data.csv')
df.head(5)

In [ ]:
df.tail(5)

In [ ]:
# 1. Setup Device
device = "mps" # Apple Silicon

# 2. Load FinBERT
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Create the sentiment pipeline
nlp = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=device)

# 3. Prepar text for FinBERT
def combine_text(row):
    headline = str(row['headline'])
    summary = str(row['summary'])
    
    # If summary is 'nan', 'NaN', or empty, just use the headline
    if not summary or summary.lower() == 'nan' or len(summary.strip()) == 0:
        return headline
    
    # Otherwise, join them (FinBERT handles up to 512 tokens)
    return f"{headline}. {summary}"

processed_text = df.apply(combine_text, axis=1).tolist()

# 4. Batch processing
batch_size = 64
results = []

for i in tqdm(range(0, len(processed_text), batch_size)):
    batch = processed_text[i : i + batch_size]
    # Truncation is critical here since adding summaries makes the text longer
    batch_results = nlp(batch, truncation=True, max_length=512)
    results.extend(batch_results)

# 5. Extract results back to DataFrame
df['sentiment_label'] = [res['label'] for res in results]
df['sentiment_score'] = [res['score'] for res in results]

In [ ]:
df.head(5)

In [ ]:
df.to_csv("market_news_with_sentiment.csv",index=False)

In [12]:
# 1. Load Data
stock_returns = pd.read_csv("stock_returns.csv")
etf_returns = pd.read_csv("etf_returns.csv")
stock_returns["date"] = pd.to_datetime(stock_returns["date"])
etf_returns["date"] = pd.to_datetime(etf_returns["date"])
stock_tickers = [c for c in stock_returns.columns if c != "date"]
etf_tickers = [c for c in etf_returns.columns if c != "date"]
tickers = stock_tickers + etf_tickers

# 2. Resample Returns & FIX ALIGNMENT
# We force everything to the 1st of the month so the indices match perfectly.
stock_monthly = (1 + stock_returns.set_index("date")[stock_tickers]).resample("M").prod() - 1
etf_monthly = (1 + etf_returns.set_index("date")[etf_tickers]).resample("M").prod() - 1
monthly_rets = pd.concat([stock_monthly, etf_monthly], axis=1).reindex(columns=tickers)

# CRITICAL FIX: Move return index from Month-End to Month-Start
monthly_rets.index = monthly_rets.index.to_period("M").to_timestamp()

# 3. Process News Sentiment
news = pd.read_csv("market_news_with_sentiment.csv", low_memory=False)
news["timestamp"] = pd.to_datetime(news["timestamp"], utc=True, errors="coerce")
news = news.dropna(subset=["timestamp"]).copy()
news["timestamp"] = news["timestamp"].dt.tz_localize(None)
news["symbols"] = news["symbols"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
news["n_symbols"] = news["symbols"].apply(lambda x: len(x) if isinstance(x, list) else 0)
news_exploded = news.explode("symbols").rename(columns={"symbols": "ticker"})
news_exploded = news_exploded[news_exploded["ticker"].isin(set(tickers))].copy()

# 4. Map Sentiment to Scores
label_sign = {"positive": 1, "negative": -1, "neutral": 0}
news_exploded["sign"] = news_exploded["sentiment_label"].map(label_sign).fillna(0)
news_exploded["sent_strength"] = news_exploded["sign"] * (2 * news_exploded["sentiment_score"] - 1)
news_exploded.loc[news_exploded["sentiment_label"].eq("neutral"), "sent_strength"] = 0.0
news_exploded["month"] = news_exploded["timestamp"].dt.to_period("M").dt.to_timestamp()
news_exploded = news_exploded[news_exploded["month"] >= "2015-01-01"].copy()
news_exploded["w"] = 1.0 / np.sqrt(news_exploded["n_symbols"].clip(lower=1))

# 5. Create Signal Matrix & Shrinkage
sent_signal = (
    news_exploded.groupby(["month", "ticker"])
    .apply(lambda g: np.average(g["sent_strength"], weights=g["w"]))
    .unstack(level=1).reindex(columns=tickers).fillna(0.0)
)
article_count = (
    news_exploded.groupby(["month", "ticker"]).size().unstack(level=1).reindex(columns=tickers).fillna(0.0)
)
k = 10.0
sent_signal = sent_signal * (article_count / (article_count + k))

# 6. Time-Series Z-Score
def expanding_zscore(series, min_periods=12):
    mu = series.expanding(min_periods=min_periods).mean().shift(1)
    sd = series.expanding(min_periods=min_periods).std(ddof=0).shift(1)
    z = (series - mu) / sd
    return z.replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(-3, 3)

sent_signal_z = sent_signal.apply(expanding_zscore)
# Smooth the z scored signal
sent_signal_z_smooth = sent_signal_z.ewm(com=2).mean()
# 7. Dynamic Beta Estimation (Regression of Sentiment vs. Future Returns)
# baseline_ret is the rolling mean of the past 12 months
baseline_ret = monthly_rets.rolling(12, min_periods=6).mean().shift(1).fillna(0.0)
# future_ret is the target (what happens next month)
future_ret = monthly_rets.shift(-1) 

def estimate_expanding_beta(signal_s, future_ret_s, min_obs=24, beta_cap=0.08):
    betas = pd.Series(index=signal_s.index, dtype=float)

    for t in range(len(signal_s.index)):
        x_hist = signal_s.iloc[:t]
        y_hist = future_ret_s.iloc[:t]

        mask = x_hist.notna() & y_hist.notna()
        x_sub = x_hist[mask]
        y_sub = y_hist[mask]

        if len(x_sub) < min_obs:
            betas.iloc[t] = 0.0
            continue

        x_centered = x_sub - x_sub.mean()
        y_centered = y_sub - y_sub.mean()

        denom = np.sum(x_centered ** 2)
        if denom <= 1e-12:
            betas.iloc[t] = 0.0
            continue

        beta = np.sum(x_centered * y_centered) / denom
        betas.iloc[t] = np.clip(beta, -beta_cap, beta_cap)

    return betas.fillna(0.0)

beta_matrix = pd.DataFrame(index=sent_signal_z.index, columns=tickers)
beta_matrix = beta_matrix.astype(float).ewm(com=3).mean()
for ticker in tickers:
    beta_matrix[ticker] = estimate_expanding_beta(sent_signal_z[ticker], future_ret[ticker])

# 8. Final Q Matrix Calculation
Q_matrix = baseline_ret.reindex_like(sent_signal_z_smooth) + (beta_matrix * sent_signal_z_smooth)

Q_matrix.head()

/var/folders/lv/z5s79n1j3fs87jp5qd8_r4dm0000gn/T/ipykernel_97565/145263295.py:12: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  stock_monthly = (1 + stock_returns.set_index("date")[stock_tickers]).resample("M").prod() - 1
/var/folders/lv/z5s79n1j3fs87jp5qd8_r4dm0000gn/T/ipykernel_97565/145263295.py:13: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  etf_monthly = (1 + etf_returns.set_index("date")[etf_tickers]).resample("M").prod() - 1
/var/folders/lv/z5s79n1j3fs87jp5qd8_r4dm0000gn/T/ipykernel_97565/145263295.py:41: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: np.average(g["sen

ticker,AAPL,AMZN,CAT,JNJ,JPM,KO,MSFT,NVDA,V,XOM,EEM,EFA,IWM,QQQ,SPY,TLT,VNQ,XLE,XLK,XLV
month,,,,,,,,,,,,,,,,,,,,
2015-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2015-02-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2015-03-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2015-04-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2015-05-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
Q_matrix.tail(10)

ticker,AAPL,AMZN,CAT,JNJ,JPM,KO,MSFT,NVDA,V,XOM,EEM,EFA,IWM,QQQ,SPY,TLT,VNQ,XLE,XLK,XLV
month,,,,,,,,,,,,,,,,,,,,
2025-03-01,0.026306,0.016379,0.004904,0.003887,0.033468,0.020159,-0.001181,0.043853,0.019655,0.009058,0.004066,0.000763,0.008269,0.011439,0.013934,-0.001406,0.008572,0.009739,-0.005321,0.005071
2025-04-01,0.022789,0.007055,-0.005942,0.004611,0.021099,0.020773,-0.008061,0.023912,0.018808,0.005372,0.004268,0.001692,-0.000802,0.003496,0.006245,-0.003237,0.003322,-0.000541,-0.013723,0.000398
2025-05-01,0.020390,0.006229,-0.004904,0.008022,0.024283,0.020263,0.003651,0.025437,0.018446,-0.005383,0.004394,0.008965,0.001284,0.007974,0.009535,0.001150,0.009840,-0.010722,-0.005271,0.000152
2025-06-01,0.005008,0.014867,0.003938,0.004305,0.026294,0.017595,0.011862,0.019318,0.023966,-0.007685,0.006668,0.009762,0.004284,0.011792,0.010736,-0.004310,0.012632,-0.008306,0.000179,-0.006159
2025-07-01,-0.000083,0.012399,0.015077,0.004819,0.034621,0.013620,0.012662,0.022509,0.022381,-0.001112,0.010649,0.013969,0.010705,0.014860,0.012257,-0.002339,0.011004,-0.003252,0.002874,-0.006129
2025-08-01,-0.003370,0.020315,0.022755,0.005251,0.032027,0.005881,0.023507,0.037387,0.021829,-0.000535,0.011397,0.010521,0.003720,0.015872,0.012882,-0.006563,0.005755,-0.002271,0.008017,-0.010748
2025-09-01,0.004978,0.021939,0.016927,0.007396,0.028463,-0.001413,0.019895,0.034094,0.019874,0.001778,0.013549,0.011881,0.012374,0.015612,0.012810,-0.007724,0.003299,0.002456,0.010108,-0.010294
2025-10-01,0.012987,0.015215,0.018004,0.012821,0.036939,-0.003253,0.019392,0.036977,0.015633,0.001186,0.015077,0.013152,0.013679,0.019163,0.014205,-0.005132,-0.000014,0.004310,0.014809,-0.007062
2025-11-01,0.020573,0.024699,0.038308,0.017963,0.031118,0.007431,0.023520,0.037402,0.012476,0.002674,0.021224,0.018679,0.018528,0.023576,0.016893,0.000785,0.000298,0.002940,0.021562,0.000504


In [14]:
Q_matrix.to_csv("Q_matrix.csv")